## 1. Install Dependencies

Run this once if your environment does not already have these packages.

In [1]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
if not requirements.exists():
    requirements = Path("llm/api/requirements.txt")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])

0

## 2. Load Environment and Model

If your Hugging Face repo is private, set `HF_TOKEN` in `.env`. If it is public, the token is optional.

In [2]:
import math
import os
import re
from collections import Counter

import torch
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from peft import PeftModel
from pypdf import PdfReader
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

load_dotenv(".env", override=True)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO = "Glccampos/llm_qween"
HF_TOKEN = os.getenv("HF_TOKEN")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device: {DEVICE}")
print(f"Adapter repo: https://huggingface.co/{ADAPTER_REPO}")

Device: cpu
Adapter repo: https://huggingface.co/Glccampos/llm_qween


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_REPO,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    dtype=DTYPE,
    trust_remote_code=True,
)
base_model.to(DEVICE)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
    token=HF_TOKEN,
)
model.eval()

print("Loaded base model and LoRA adapter.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded base model and LoRA adapter.


## 3. Wrap the Model with LangChain

The adapter was trained with chat-style prompts, so we format each request with Qwen's chat template before passing it to the text-generation pipeline.

In [4]:
USE_SAMPLING = True
TEMPERATURE = 0.1
TOP_P = 0.9
TOP_K = 50

# Temperature/top-p/top-k only make sense when sampling is enabled.
generation_kwargs = {
    "do_sample": USE_SAMPLING,
    "max_new_tokens": 128,
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

if USE_SAMPLING:
    generation_kwargs.update(
        {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "top_k": TOP_K,
        }
    )
else:
    # Neutral defaults keep deterministic decoding free of sampling warnings.
    generation_kwargs.update(
        {
            "temperature": 0.1,
            "top_p": 1.0,
            "top_k": 50,
        }
    )

model.generation_config.update(**generation_kwargs)
model.generation_config.max_length = None

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    clean_up_tokenization_spaces=False,
    device=0 if DEVICE == "cuda" else -1,
)
text_generation_pipeline.generation_config.update(**generation_kwargs)
text_generation_pipeline.generation_config.max_length = None

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline,
    pipeline_kwargs=generation_kwargs,
)


def format_qa_prompt(inputs):
    context = inputs.get("context", "")
    question = inputs["question"]
    user_prompt = (
        "Answer the question using only the context when context is provided.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}"
    )
    messages = [
        {"role": "system", "content": "You are a helpful Q&A assistant."},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


qa_chain = RunnableLambda(format_qa_prompt) | llm | StrOutputParser()

print("LangChain Q&A chain is ready.")

LangChain Q&A chain is ready.


## 4. Build a RAG Chain from the PDF

In [5]:
import math
import re
from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

PDF_PATH = "Analytics Engineer .pdf"

reader = PdfReader(PDF_PATH)
pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)
chunks = text_splitter.split_text(pdf_text)


def tokenize_for_retrieval(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


chunk_term_counts = [Counter(tokenize_for_retrieval(chunk)) for chunk in chunks]
document_frequencies = Counter(
    term for counts in chunk_term_counts for term in counts.keys()
)


def retrieve_context(question, top_k=3):
    query_terms = tokenize_for_retrieval(question)
    if not query_terms:
        return "\n\n".join(chunks[:top_k])

    scored_chunks = []
    for index, (chunk, term_counts) in enumerate(zip(chunks, chunk_term_counts)):
        score = 0.0
        for term in query_terms:
            if term not in term_counts:
                continue
            idf = math.log((len(chunks) + 1) / (document_frequencies[term] + 1)) + 1
            score += term_counts[term] * idf
        scored_chunks.append((score, index, chunk))

    best_chunks = [
        chunk
        for score, _, chunk in sorted(scored_chunks, reverse=True)[:top_k]
        if score > 0
    ]
    return "\n\n---\n\n".join(best_chunks or chunks[:top_k])


def answer_from_pdf(question):
    context = retrieve_context(question)
    return qa_chain.invoke({"context": context, "question": question}).strip()


print(f"Loaded {len(chunks)} chunk(s) from {PDF_PATH}")

Loaded 4 chunk(s) from Analytics Engineer .pdf


## 5. Ask Questions About the PDF

In [6]:
question = "What skills are required for the Analytics Engineer role?"
answer = answer_from_pdf(question)
print(answer)

Hands-on experience with dbt, ETL optimization, version control tools such as GitHub or GitLab, and visualization tools such as Sigma or Power BI.


## 6. Connect LangChain to the Local Trino MCP Server

This starts your local `mcp/trino_mcp.py` server over stdio and loads its tools into LangChain.

**Windows Jupyter + WSL:** the notebook connects over **SSE** to `http://127.0.0.1:8765/sse`.

**MCP Inspector** (`npx @modelcontextprotocol/inspector …`) is **not** enough — it uses stdio (ports 6274/6277), not port 8765.

In a **second** WSL terminal (leave it running):

```bash
cd /mnt/c/Users/guslc/project/mcp
source .venv/bin/activate
python trino_mcp.py --transport sse --port 8765
```

You should see: `SSE endpoint: http://127.0.0.1:8765/sse`. Then run the cell below.

Optional in `llm/api/.env`: `TRINO_MCP_SSE_URL=http://127.0.0.1:8765/sse`

**Linux Jupyter kernel:** uses stdio with `mcp/.venv/bin/python` (no SSE server needed).

In [7]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(".env", override=True)

# Windows Jupyter cannot spawn stdio MCP (fileno / asyncio). Use SSE instead:
#   WSL terminal: cd /mnt/c/Users/guslc/project/mcp && .venv/bin/python trino_mcp.py --transport sse
#   Then set TRINO_MCP_SSE_URL=http://127.0.0.1:8765/sse in llm/api/.env

print(f"platform={sys.platform}")
print(f"cwd={Path.cwd()}")
print(f"python={sys.executable}")


def _find_trino_mcp_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates: list[Path] = []
    if env_dir := os.getenv("TRINO_MCP_DIR"):
        candidates.append(Path(env_dir))
    candidates.extend(
        [
            cwd / ".." / ".." / "mcp",  # llm/api -> project/mcp
            cwd / "mcp",
        ]
    )
    if sys.platform == "linux":
        candidates.append(Path("/mnt/c/Users/guslc/project/mcp"))
    elif sys.platform == "win32":
        candidates.append(Path(r"C:\Users\guslc\project\mcp"))

    for path in candidates:
        resolved = path.resolve()
        if (resolved / "trino_mcp.py").exists():
            return resolved
    tried = ", ".join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(
        f"Could not find mcp/trino_mcp.py. Tried: {tried}. "
        "On WSL, run the notebook with a Linux kernel and cwd under /mnt/c/.../project. "
        "Or set TRINO_MCP_DIR=/mnt/c/Users/guslc/project/mcp in .env"
    )


def _path_usable(path: Path) -> bool:
    try:
        return path.is_file()
    except OSError:
        return False


def _check_sse_reachable(url: str) -> None:
    import socket
    from urllib.parse import urlparse

    parsed = urlparse(url)
    host = parsed.hostname or "127.0.0.1"
    port = parsed.port or 8765
    try:
        with socket.create_connection((host, port), timeout=3):
            pass
    except OSError as exc:
        raise ConnectionError(
            f"Nothing is listening on {host}:{port} ({exc}).\n"
            "MCP Inspector (npx … inspector) uses stdio on ports 6274/6277 — "
            "that is NOT the same as the SSE server the notebook needs.\n\n"
            "In a separate WSL terminal (keep it running):\n"
            "  cd /mnt/c/Users/guslc/project/mcp\n"
            "  source .venv/bin/activate\n"
            "  python trino_mcp.py --transport sse --port "
            f"{port}\n\n"
            f"Then re-run this cell. URL: {url}"
        ) from exc


def _build_trino_mcp_connection(mcp_dir: Path) -> dict:
    """stdio on Linux; SSE on Windows Jupyter (stdio subprocess fails with fileno)."""
    sse_url = (os.getenv("TRINO_MCP_SSE_URL") or "").strip()
    mcp_port = os.getenv("MCP_PORT", "8765")
    default_sse = f"http://127.0.0.1:{mcp_port}/sse"

    if sys.platform == "win32":
        url = sse_url or default_sse
        print(f"Windows Jupyter: SSE -> {url}")
        _check_sse_reachable(url)
        return {"transport": "sse", "url": url, "timeout": 30.0}

    if sse_url:
        return {"transport": "sse", "url": sse_url, "timeout": 30.0}

    server = mcp_dir / "trino_mcp.py"
    linux_python = mcp_dir / ".venv" / "bin" / "python"
    if not _path_usable(linux_python):
        raise FileNotFoundError(
            f"Missing {linux_python}. Run: cd {mcp_dir} && uv sync"
        )
    return {
        "transport": "stdio",
        "command": str(linux_python),
        "args": [str(server)],
        "cwd": str(mcp_dir),
    }


TRINO_MCP_DIR = _find_trino_mcp_dir()
_trino_connection = _build_trino_mcp_connection(TRINO_MCP_DIR)

print(f"TRINO_MCP_DIR={TRINO_MCP_DIR}")
print(f"MCP transport={_trino_connection.get('transport')}")

mcp_client = MultiServerMCPClient({"trino": _trino_connection})

trino_tools = await mcp_client.get_tools()
trino_tool_names = [tool.name for tool in trino_tools]
trino_tool_names

platform=win32
cwd=c:\Users\guslc\project\llm\api
python=c:\Users\guslc\miniconda3\envs\llm\python.exe
Windows Jupyter: SSE -> http://127.0.0.1:8765/sse
TRINO_MCP_DIR=C:\Users\guslc\project\mcp
MCP transport=sse


['query', 'list_catalogs', 'list_schemas', 'list_tables', 'describe_table']

In [8]:
import json


def _normalize_mcp_result(value):
    """Unwrap LangChain MCP tool output into the dict returned by trino_mcp.py."""
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {"text": value}

    if isinstance(value, list):
        if len(value) == 1:
            item = value[0]
            if isinstance(item, dict) and "text" in item:
                return _normalize_mcp_result(item["text"])
            return _normalize_mcp_result(item)
        return {"rows": value, "row_count": len(value)}

    return {"value": value}


async def call_trino_tool(name: str, arguments: dict | None = None):
    """Call a Trino MCP tool by name from the loaded LangChain tools."""
    arguments = arguments or {}
    matches = [tool for tool in trino_tools if tool.name == name or tool.name.endswith(name)]
    if not matches:
        raise ValueError(f"Tool {name!r} not found. Available tools: {trino_tool_names}")
    raw_result = await matches[0].ainvoke(arguments)
    return _normalize_mcp_result(raw_result)


catalogs = await call_trino_tool("list_catalogs")
catalogs

{'columns': ['Catalog'],
 'rows': [['clickhouse'], ['iceberg'], ['system'], ['tpcds'], ['tpch']],
 'row_count': 5,
 'max_rows': 1000}

In [9]:
schemas = await call_trino_tool("list_schemas", {"catalog": "iceberg"})
schemas

{'columns': ['Schema'],
 'rows': [['default'], ['forecast'], ['information_schema']],
 'row_count': 3,
 'max_rows': 1000}

## 7. Ask Trino Questions with Natural Language

`ask_trino(...)` uses the MCP tools to inspect the selected Trino schema, asks the LangChain model to write SQL, executes that SQL through the MCP `query` tool, and summarizes the result.

In [19]:
import re


def _rows_to_markdown(columns, rows, max_rows=20):
    rows = rows[:max_rows]
    if not columns:
        return str(rows)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    return "\n".join([header, separator, *body])    


def _extract_sql(model_output: str) -> str:
    text = model_output.strip()
    fenced = re.search(r"```(?:sql)?\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()
    text = re.sub(r"^SQL\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    text = text.rstrip(";").strip()

    lowered = text.lower()
    allowed_prefixes = ("select", "with", "show", "describe", "desc")
    forbidden_words = ("insert", "update", "delete", "drop", "alter", "create", "truncate", "merge")
    if not lowered.startswith(allowed_prefixes):
        raise ValueError(f"Model did not produce a read-only SQL query: {model_output!r}")
    if any(re.search(rf"\b{word}\b", lowered) for word in forbidden_words):
        raise ValueError(f"Refusing to execute non-read-only SQL: {text}")
    # Model sometimes repeats the query; keep the first SELECT/WITH only.
    starts = list(re.finditer(r"(?is)\b(select|with)\b", text))
    if len(starts) > 1:
        text = text[starts[0].start() : starts[1].start()].strip()
    return text.rstrip(";").strip()


def _normalize_json_sql(sql: str) -> str:
    """Fix common LLM mistake: comparing whole JSON column to a small JSON literal."""
    fixed = sql
    literal_match = re.search(
        r"(?is)\b(request_json|response_json)\s*=\s*'\{.*?\"([\w_]+)\"\s*:\s*\"([^\"]*)\".*?\}'",
        fixed,
    )
    if literal_match:
        col, field, value = literal_match.groups()
        replacement = f"json_extract_scalar({col}, '$.{field}') = '{value}'"
        fixed = fixed[: literal_match.start()] + replacement + fixed[literal_match.end() :]
    if re.search(r"(?i)\bcount\s*\(\s*client_id\s*\)", fixed) and "distinct" not in fixed.lower():
        fixed = re.sub(r"(?i)count\s*\(\s*client_id\s*\)", "COUNT(DISTINCT client_id)", fixed, count=1)
    return fixed.strip()


PREDICTION_JSON_SAMPLES = """
request_json example:
{"id_cliente": "292659", "id_contrato": null, "tipo_contrato": "Cash loans", "status_contrato": "ativo", "tipo_pagamento": "boleto", "finalidade_emprestimo": "compra_veiculo", "tipo_cliente": "pessoa_fisica", "tipo_portfolio": "varejo", "tipo_produto": "credito_pessoal", "categoria_bem": "automovel", "setor_vendedor": "digital", "canal_venda": "online", "faixa_rendimento": null, "combinacao_produto": null, "area_venda": null, "dia_semana_solicitacao": "SATURDAY", "data_nascimento": "1997-04-08", "data_decisao": "2024-01-01", "data_liberacao": null, "data_primeiro_vencimento": null, "data_ultimo_vencimento_original": null, "data_ultimo_vencimento": null, "data_encerramento": null, "valor_solicitado": 0.0, "valor_credito": 439740.0, "valor_bem": 315000.0, "valor_parcela": 23985.0, "valor_entrada": 0.0, "percentual_entrada": 0.0, "qtd_parcelas_planejadas": 12, "taxa_juros_padrao": 0.03, "taxa_juros_promocional": 0.03, "hora_solicitacao": 14, "flag_ultima_solicitacao_contrato": 0, "flag_ultima_solicitacao_dia": 0, "acompanhantes_cliente": 0, "flag_seguro_contratado": 0, "motivo_recusa": null, "renda_anual": 157500.0, "qtd_membros_familia": 1, "possui_carro": "N", "possui_imovel": "Y"}

response_json example:
{"request_id": "02e60220-9a7e-418f-abc8-251c8111dfb3", "probability": 0.9504, "threshold_decision": "Negado", "status": "success"}
"""


async def get_trino_schema_context(catalog="iceberg", schema="forecast", max_tables=20):
    table_result = await call_trino_tool("list_tables", {"schema_name": f"{catalog}.{schema}"})
    table_names = [row[0] for row in table_result.get("rows", [])]

    table_descriptions = []
    for table_name in table_names[:max_tables]:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        description = await call_trino_tool("describe_table", {"table": full_table_name})
        columns = []
        for row in description.get("rows", []):
            if len(row) >= 2 and row[0]:
                columns.append(f"{row[0]} {row[1]}")
        table_descriptions.append(f"{full_table_name}: " + ", ".join(columns))

    return "\n".join(table_descriptions) + "\n\n" + PREDICTION_JSON_SAMPLES


def _summarize_result(question: str, result: dict) -> str:
    columns = result.get("columns") or []
    rows = result.get("rows") or []
    if not rows:
        return "The query returned no rows."
    if len(rows) == 1 and len(rows[0]) == 1:
        value = rows[0][0]
        label = columns[0] if columns else "result"
        return f"Answer: {value} ({label})."
    preview = _rows_to_markdown(columns, rows, max_rows=5)
    return f"Result preview:\n{preview}"


async def generate_trino_sql(question: str, catalog="iceberg", schema="forecast"):
    schema_context = await get_trino_schema_context(catalog=catalog, schema=schema)
    messages = [
        {
            "role": "system",
            "content": (
                "You write read-only Trino SQL. Return exactly one SQL query, no markdown. "
                "Use fully qualified table names from the schema context. "
                "prediction_events: request_json = HTTP request body (VARCHAR JSON); "
                "response_json = model output (VARCHAR JSON) with keys like threshold_decision, probability, status. "
                "NEVER compare request_json/response_json to a JSON literal string. "
                "ALWAYS filter with json_extract_scalar(column, '$.field') = 'value'. "
                "Use COUNT(DISTINCT client_id) when counting clients. "
                "Example: SELECT COUNT(DISTINCT client_id) FROM iceberg.forecast.prediction_events "
                "WHERE json_extract_scalar(response_json, '$.threshold_decision') = 'Negado'"
            ),
        },
        {
            "role": "user",
            "content": (
                f"Schema context:\n{schema_context}\n\n"
                f"Question: {question}\n\n"
                "Write the Trino SQL query."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return _normalize_json_sql(_extract_sql(llm.invoke(prompt)))


async def ask_trino(question: str, catalog="iceberg", schema="forecast", max_rows=100):
    sql = await generate_trino_sql(question, catalog=catalog, schema=schema)
    result = await call_trino_tool("query", {"sql": sql, "max_rows": max_rows})

    preview = _rows_to_markdown(result.get("columns", []), result.get("rows", []))
    messages = [
        {"role": "system", "content": "Summarize Trino query results clearly and briefly."},
        {
            "role": "user",
            "content": (
                f"Question: {question}\n"
                f"SQL: {sql}\n\n"
                f"Result preview:\n{preview}\n\n"
                "Answer the question using the SQL result."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = llm.invoke(prompt).strip()
    if not answer:
        answer = _summarize_result(question, result)
    print("\n--- Answer ---\n", answer)
    return {"question": question, "sql": sql, "result": result, "answer": answer}

In [20]:
response = await ask_trino(
    "How many clients have their response threshold_decision=Negado?",
    catalog="iceberg",
    schema="forecast",
)

print("--- SQL ---\n", response["sql"])
print("\n--- Answer ---\n", response["answer"])
response["answer"]


--- Answer ---
 39999
--- SQL ---
 SELECT COUNT(DISTINCT client_id)
FROM iceberg.forecast.prediction_events
WHERE json_extract_scalar(response_json, '$.threshold_decision') = 'Negado'

--- Answer ---
 39999


'39999'